### LLM Call

In [2]:
import os
from openai import OpenAI
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv() 


True

In [34]:
USE_LOCAL = True  # set to False to use OpenAI

if USE_LOCAL:
    from langchain_ollama import ChatOllama
    llm_local = ChatOllama(model="gpt-oss:120b-cloud", base_url="http://localhost:11434")
    print(f"Using Ollama model: {llm_local.model}")
else:
    from langchain_openai import ChatOpenAI
    llm_local = ChatOpenAI(model="gpt-5-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
    # print(f"Using OpenAI model: {llm_local.model}")

# llm_local.invoke("Tell a fun fact").content

Using Ollama model: gpt-oss:120b-cloud


### Prompt vs Messages:

<u>**Prompt**</u> – a single raw string sent to a language‑model API that expects plain text (e.g., what is a bank?).

<u>**Message**</u> – an element of a message list that includes a role (system, human, assistant, etc.) and content. Chat‑style models (GPT‑3.5‑turbo, Claude‑chat) require a sequence of such messages.

So, prompt = one text block; message = one role‑tagged part of a conversation, and a chat model receives many messages together

In [5]:
# Messsage Example
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
my_msg = [SystemMessage(content="You are Geography expert"),
          HumanMessage(content="Tell  a fun fact")]
llm_local.invoke(my_msg).content

'Here’s a quirky geography gem you might not have heard before:\n\n**The world’s “floating” island that isn’t an island at all –\u202fLake Titicaca’s “Island of the Sun” appears to drift!**\n\n- **Where it is:** On the border of Peru and Bolivia, high up in the Andes (about 3,800\u202fm or 12,500\u202fft above sea level).  \n- **Why it’s “floating”:** The Island of the Sun (Isla del Sol) sits on the surface of Lake Titicaca, the highest navigable lake in the world. On calm days the water’s surface is so calm that the island seems to hover, almost like a floating platform.  \n- **Bonus fun fact:** The island is considered sacred by the ancient Inca and pre‑Inca cultures. According to legend, it is the birthplace of the first Inca ruler, Manco Cápac, and the site where the sun god Inti first emerged. The island’s stone ruins, terraces, and traditional reed boats (called *caballitos de totora*) give you a glimpse into a culture that literally lives on “water‑above‑the‑clouds.”  \n\nSo nex

### Prompts - and types of Prompts

<p>Prompts are concise inputs that guide an AI model toward generating specific, relevant responses. By framing the request clearly—often with context, constraints, or format cues—you shape the model’s behavior and improve output quality. Well‑crafted prompts turn raw language models into practical, task‑focused tools.</p>

<u>**PromptTemplate**</u> – formats a single string for a standard LLM. It takes input variables, substitutes them into a template, and returns one text prompt.

<u>**ChatPromptTemplate**</u> – builds a list of messages (system, human, AI, etc.) for chat‑style models. It can handle multiple roles and maintains a conversational sequence, producing a messages list rather than a single string.

<i>In short: use PromptTemplate for plain text completions; use ChatPromptTemplate when the model expects a chat/message history</i>

In [6]:
from langchain_core.prompts import PromptTemplate
user_ip = input("Enter user message/prompt here")
dynamic_prompt = PromptTemplate.from_template("Write  2-3 lines about {topic}")
final_prompt = dynamic_prompt.invoke({"topic":user_ip})
llm_local.invoke(final_prompt).content

In [7]:
from langchain_core.prompts import ChatPromptTemplate
user_ip = input("Enter user message/prompt here")
prompt_template = ChatPromptTemplate.from_messages([
                    ("system","""You are a helpful assistant in Indian History. Follow these rules when answering:
                    - Use only standard ASCII letters, digits, and punctuation. Use normal spaces only (no special Unicode spaces or symbols).
                    - Do not use characters like narrow no-break space (U+202F) or other non-ASCII spacing/symbols.
                    - Prefer simple, clear prose. Use markdown only if the user asks for it."""),
                    ("user","Write a important fact about {topic}")
])

final_prompt = prompt_template.invoke({"topic":user_ip})
res=llm_local.invoke(final_prompt)
res.content

### Introducing Pydantic - Why we need Pydantic in Langchain ? 

Often times LLM may produce outputs that vary too much in terms of structure, and if we need structured output ,we can use Pydantic.
Pydantic provides typed, validated data structures that LangChain’s prompt‑/message‑templates work with.

<u>**Schema enforcement**</u> – PromptTemplate, ChatPromptTemplate, and output parsers declare the variables they expect (input_variables, messages, etc.) as Pydantic fields. When you call format or format_messages, Pydantic checks that all required keys are present and that their types match, raising clear errors instead of silent string‑substitution bugs.

<u>**Serialization / deserialization**</u> – Templates and chains can be saved to JSON/YAML and later re‑loaded; Pydantic’s .model_dump() / .model_construct() handle this automatically.

<u>**Integration with LangChain models**</u> – Many LangChain components (e.g., LLMChain, AgentExecutor, output parsers) accept and return Pydantic models, enabling easy conversion between raw model output and structured Python objects.

<u>**Developer ergonomics**</u> – IDEs get autocomplete and type‑checking for the fields defined in a Pydantic model, reducing runtime mistakes.

In short, Pydantic gives LangChain type safety, validation, and easy (de)serialization for prompts, messages, and the data flowing through LLM pipelines.

#### - Guided Prompts

In [12]:
# One way of getting results in specific schema is to prompt the LLM with instructions , see below example

prompt_guided = PromptTemplate.from_template("Tell a joke. Generate output in key-value pair")
# print(prompt_guided.template)
results = llm_local.invoke(prompt_guided.template).content
results

'```json\n{\n  "setup": "Why do programmers prefer dark mode?",\n  "punchline": "Because light attracts bugs!"\n}\n```'

#### - Pydantic Way of generating Structure Outputs

In [26]:
# Now lets see how Pydantic achieves forced structure/schema

from pydantic import BaseModel, Field
# Step - 1: Create a schema by defining BaseModel schema 
class llm_schema(BaseModel):
    setup: str = Field(description = "The setup for joke")
    punchline: str = Field(description = "The punchline for joke")


In [22]:
# Just for check -  lets create an object
obj = llm_schema(**{"setup" : "example setup" , "punchline": "example punchline"})
obj

# If we pass some other key, or some other datatype which is different from BaseModel , this will throw error, uncomment below to see that
# obj = llm_schema(**{"setup_1" : "example setup" , "punchline": "example punchline"})



llm_schema(setup='example setup', punchline='example punchline')

In [36]:
# Create s structure llm model and then invoke it with the same prompt as before, see below
llm_local_structured_output = llm_local.with_structured_output(llm_schema)
llm_local_structured_output.invoke("Tell a joke.Tell a joke. Return as JSON with 'setup' and 'punchline' keys.")

llm_schema(setup='Why did the scarecrow become a successful neurosurgeon?', punchline='Because he was outstanding in his field and always had a lot of patients!')